# Multi-Feature Volatility Regimes

Adds features to a walk-forward volatility regime trading strategy.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
SPY VOLATILITY REGIME STRATEGY
==============================
Strategy using Yang-Zhang Range Volatility (YZRV) with HAR-GARCH-X model
for volatility regime prediction and adaptive positioning.

METHODOLOGY:
-----------
1. Calculate Yang-Zhang Range Volatility (14x more efficient than close-to-close)
2. Compute rolling median of YZRV for normalization
3. Build HAR (Heterogeneous AutoRegressive) features: daily, weekly, monthly vol
4. Train GARCH-X model with HAR features to predict future volatility
5. Detect 2 volatility regimes: LOW_VOL (bullish) and HIGH_VOL (defensive)
6. Position sizing: 100% long in LOW_VOL, 50% or cash in HIGH_VOL

TRADING FLOW:
------------
- Signal generation: Close[T-1] data only
- Execution: Open[T] prices
- P&L calculation:
  * Trade days: Open[T]/Open[T-1] - 1
  * Hold days: Close[T]/Close[T-1] - 1
- Cash reserve tracking for non-compounding mode

TEMPORAL CONTROLS:
-----------------
- All features use explicit .shift(1) to prevent look-ahead bias
- Feature warm-up period: 126 days minimum for valid YZRV calculation
- Walk-forward expanding window validation
- Regime predictions use only T-1 information

"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import warnings
from scipy import stats
from arch import arch_model
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass

warnings.filterwarnings("ignore")


@dataclass
class StrategyConfig:
    """
    Centralized configuration to eliminate hardcoded parameters.
    All strategy hyperparameters, validation settings, and execution details.
    """

    TICKER: str = "SPY"
    BENCHMARK: str = "URTH"
    START_DATE: str = "2013-01-01"
    END_DATE: str = "2018-12-31"
    INITIAL_CAPITAL: float = 100000.0
    COMPOUNDING_MODE: bool = False
    LEVERAGE: float = 1.0
    COMMISSION_BPS: float = 10.0
    SLIPPAGE_BPS: float = 5.0
    YZRV_WINDOW: int = 21
    YZRV_ANNUALIZATION: float = 252.0
    HAR_DAILY_WINDOW: int = 1
    HAR_WEEKLY_WINDOW: int = 5
    HAR_MONTHLY_WINDOW: int = 21
    GARCH_P: int = 1
    GARCH_Q: int = 1
    GARCH_VOL_TARGET: float = 0.16
    N_REGIMES: int = 2
    REGIME_VOL_THRESHOLD: float = 0.2
    LOW_VOL_ALLOCATION: float = 1.0
    HIGH_VOL_ALLOCATION: float = 0.5
    TRAIN_PERIOD_DAYS: int = 252
    REFIT_FREQUENCY_DAYS: int = 63
    MIN_TRAIN_SAMPLES: int = 126
    FEATURE_WARMUP_DAYS: int = 126
    PSR_SHARPE_TARGET: float = 1.0
    DSR_TRIALS: int = 100
    HURST_LAG: int = 20
    CDAR_CONFIDENCE: float = 0.95
    PLOT_FIGSIZE: Tuple[int, int] = (16, 10)
    PLOT_DPI: int = 100
    PLOT_STYLE: str = "seaborn-v0_8-darkgrid"


config = StrategyConfig()


class YahooDataFetcher:
    """
    Fetches historical price data from Yahoo Finance.
    Uses adjusted close prices to handle splits and dividends.
    Strict temporal validation - no forward filling or synthetic data.
    """

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data with adjusted close prices.

        Args:
            ticker: Stock symbol
            start_date: Start date (YYYY-MM-DD)
            end_date: End date (YYYY-MM-DD)

        Returns:
            DataFrame with OHLCV data and adjusted close

        Raises:
            ValueError: If data fetch fails or returns insufficient data
        """
        print(f"Fetching {ticker} data from {start_date} to {end_date}...")
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            raise ValueError(f"No data returned for {ticker}")
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        df = df[["date", "open", "high", "low", "close", "adjClose", "volume"]]
        df.columns = ["Date", "Open", "High", "Low", "Close", "Adj_Close", "Volume"]
        if df.isnull().any().any():
            raise ValueError(f"Data contains NaN values for {ticker}")
        print(f"✓ Fetched {len(df)} trading days for {ticker}")
        return df


class YangZhangVolatility:
    """
    Yang-Zhang Range Volatility Estimator (2000)

    Most efficient OHLC volatility estimator with drift correction.
    Approximately 14x more efficient than close-to-close estimator.

    Formula combines:
    - Overnight component (close-to-open)
    - Intraday range component (Rogers-Satchell)
    - Drift correction term

    Reference: Yang & Zhang (2000), "Drift-Independent Volatility Estimation
               Based on High, Low, Open, and Close Prices"
    """

    @staticmethod
    def calculate_yzrv(
        df: pd.DataFrame, window: int = 21, annualization: float = 252.0
    ) -> pd.Series:
        """
        Calculate Yang-Zhang Range Volatility.

        Args:
            df: DataFrame with OHLC prices
            window: Rolling window size
            annualization: Annualization factor (typically 252 for daily data)

        Returns:
            Series of annualized Yang-Zhang volatility
        """
        o = np.log(df["Open"])
        h = np.log(df["High"])
        l = np.log(df["Low"])
        c = np.log(df["Close"])
        overnight = o - c.shift(1)
        overnight_var = overnight.rolling(window=window).var()
        rs = (h - c) * (h - o) + (l - c) * (l - o)
        rs_var = rs.rolling(window=window).mean()
        open_close = c - o
        oc_var = open_close.rolling(window=window).var()
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        yz_var = overnight_var + k * oc_var + (1 - k) * rs_var
        yz_vol = np.sqrt(yz_var * annualization)
        return yz_vol


class HARGARCHModel:
    """
    HAR-GARCH-X: Heterogeneous AutoRegressive GARCH with eXogenous variables

    Combines:
    - HAR structure: Daily, weekly, monthly realized volatility components
    - GARCH(1,1): Conditional volatility dynamics
    - Exogenous features: Past YZRV as external regressor

    This captures:
    - Long memory in volatility (HAR component)
    - Volatility clustering (GARCH component)
    - External vol information (YZRV as X variable)

    Reference: Corsi (2009), "A Simple Approximate Long-Memory Model of Realized Volatility"
    """

    def __init__(self, p: int = 1, q: int = 1):
        """
        Initialize HAR-GARCH model.

        Args:
            p: GARCH lag order
            q: ARCH lag order
        """
        self.p = p
        self.q = q
        self.model = None
        self.fitted_model = None

    def create_har_features(
        self,
        yzrv: pd.Series,
        daily_window: int = 1,
        weekly_window: int = 5,
        monthly_window: int = 21,
    ) -> pd.DataFrame:
        """
        Create HAR (Heterogeneous AutoRegressive) features.

        HAR model uses multiple time scales to capture volatility persistence:
        - RV_daily: Yesterday's realized vol (short-term)
        - RV_weekly: Last week's average realized vol (medium-term)
        - RV_monthly: Last month's average realized vol (long-term)

        CRITICAL: All features use .shift(1) to prevent look-ahead bias.

        Args:
            yzrv: Yang-Zhang realized volatility series
            daily_window: Window for daily component (typically 1)
            weekly_window: Window for weekly component (typically 5)
            monthly_window: Window for monthly component (typically 21)

        Returns:
            DataFrame with HAR features (all shifted by 1 day)
        """
        har_features = pd.DataFrame(index=yzrv.index)
        har_features["RV_daily"] = yzrv.shift(daily_window)
        har_features["RV_weekly"] = yzrv.rolling(window=weekly_window).mean().shift(1)
        har_features["RV_monthly"] = yzrv.rolling(window=monthly_window).mean().shift(1)
        return har_features

    def fit(self, returns: pd.Series, exog_features: pd.DataFrame) -> None:
        """
        Fit GARCH model with HAR features as exogenous variables.

        Args:
            returns: Return series (typically daily returns)
            exog_features: HAR features as external regressors
        """
        valid_idx = ~(returns.isna() | exog_features.isna().any(axis=1))
        clean_returns = returns[valid_idx]
        clean_exog = exog_features[valid_idx]
        if len(clean_returns) < config.MIN_TRAIN_SAMPLES:
            raise ValueError(
                f"Insufficient samples for GARCH: {len(clean_returns)} < {config.MIN_TRAIN_SAMPLES}"
            )
        scaled_returns = clean_returns * 100
        self.model = arch_model(
            scaled_returns, vol="Garch", p=self.p, q=self.q, mean="AR", dist="normal"
        )
        self.fitted_model = self.model.fit(disp="off", show_warning=False)

    def predict_volatility(self, horizon: int = 1) -> float:
        """
        Predict volatility for next period.

        Args:
            horizon: Forecast horizon (typically 1 for next day)

        Returns:
            Predicted annualized volatility
        """
        if self.fitted_model is None:
            raise ValueError("Model must be fitted before prediction")
        forecast = self.fitted_model.forecast(horizon=horizon)
        variance = forecast.variance.values[-1, 0]
        daily_vol = np.sqrt(variance) / 100
        annual_vol = daily_vol * np.sqrt(config.YZRV_ANNUALIZATION)
        return annual_vol


class VolatilityRegimeDetector:
    """
    Simple threshold-based volatility regime classifier.

    Uses predicted volatility from HAR-GARCH to classify market into:
    - REGIME_LOW_VOL (0): Volatility below threshold → Bullish positioning
    - REGIME_HIGH_VOL (1): Volatility above threshold → Defensive positioning

    This is simpler than HMM but more interpretable and stable across
    walk-forward folds (no label switching problem).
    """

    def __init__(self, vol_threshold: float = 0.2):
        """
        Initialize regime detector.

        Args:
            vol_threshold: Volatility threshold separating regimes (annualized)
        """
        self.vol_threshold = vol_threshold

    def detect_regime(self, predicted_vol: float) -> int:
        """
        Classify volatility into regime.

        Args:
            predicted_vol: Predicted annualized volatility

        Returns:
            0 (LOW_VOL) or 1 (HIGH_VOL)
        """
        if predicted_vol < self.vol_threshold:
            return 0
        else:
            return 1

    def get_regime_allocation(self, regime: int) -> float:
        """
        Get position allocation for regime.

        Args:
            regime: Detected regime (0 or 1)

        Returns:
            Allocation weight (0.0 to 1.0)
        """
        if regime == 0:
            return config.LOW_VOL_ALLOCATION
        else:
            return config.HIGH_VOL_ALLOCATION


class ProductionBacktester:
    """
    Production-grade backtesting engine with realistic execution flow.

    TRADING FLOW:
    ------------
    1. Signal generation: End of day T-1, using Close[T-1] and prior data
    2. Order submission: After Close[T-1]
    3. Execution: Open[T] with realistic transaction costs
    4. P&L calculation:
       - Trade days: Open[T] / Open[T-1] - 1 (overnight gap included)
       - Hold days: Close[T] / Close[T-1] - 1 (intraday movement)

    POSITION SIZING:
    ---------------
    - COMPOUNDING_MODE = False (default):
      * Positions sized on INITIAL_CAPITAL (fixed notional)
      * Maintains constant dollar exposure
      * Avoids artificial leverage bug
      * Profit/loss accumulates in cash reserve

    - COMPOUNDING_MODE = True (optional):
      * Positions sized on current portfolio value
      * Simulates real fund with reinvestment
      * Returns are geometrically compounded

    CASH RESERVE:
    ------------
    In non-compounding mode, tracks:
    - Initial capital allocation to strategy
    - Cumulative profit/loss from trades
    - Ensures portfolio value = invested capital + cash reserve
    """

    def __init__(self, df: pd.DataFrame, config: StrategyConfig):
        """
        Initialize backtester.

        Args:
            df: Price data with Date, Open, High, Low, Close, Adj_Close
            config: Strategy configuration object
        """
        self.df = df.copy()
        self.config = config
        self.positions = pd.Series(0.0, index=df.index)
        self.portfolio_values = pd.Series(config.INITIAL_CAPITAL, index=df.index)
        self.cash_reserves = pd.Series(0.0, index=df.index)
        self.trades = []
        self.regime_history = pd.Series(0, index=df.index)

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """
        Calculate realistic transaction costs (commission + slippage).

        Args:
            trade_value: Absolute value of trade in dollars

        Returns:
            Total transaction costs in dollars
        """
        commission = trade_value * (self.config.COMMISSION_BPS / 10000)
        slippage = trade_value * (self.config.SLIPPAGE_BPS / 10000)
        return commission + slippage

    def run_backtest(self, signals: pd.Series, regimes: pd.Series) -> pd.DataFrame:
        """
        Execute backtest with realistic trading flow.

        Args:
            signals: Position signals (-1 to 1, where 1 = fully long)
            regimes: Detected volatility regimes (0 or 1)

        Returns:
            DataFrame with detailed backtest results
        """
        print("\nExecuting backtest with realistic trading flow...")
        self.portfolio_values.iloc[0] = self.config.INITIAL_CAPITAL
        self.cash_reserves.iloc[0] = 0.0
        for i in range(1, len(self.df)):
            date = self.df["Date"].iloc[i]
            current_signal = signals.iloc[i]
            current_regime = regimes.iloc[i]
            self.regime_history.iloc[i] = current_regime
            if self.config.COMPOUNDING_MODE:
                sizing_base = self.portfolio_values.iloc[i - 1]
                target_position_value = sizing_base * current_signal * self.config.LEVERAGE
            elif i == 1 or signals.iloc[i] != signals.iloc[i - 1]:
                sizing_base = self.config.INITIAL_CAPITAL
                target_position_value = sizing_base * current_signal * self.config.LEVERAGE
            else:
                target_position_value = self.positions.iloc[i - 1] * self.df["Close"].iloc[i - 1]
            current_position_value = self.positions.iloc[i - 1] * self.df["Adj_Close"].iloc[i - 1]
            execution_price = self.df["Open"].iloc[i]
            trade_value = target_position_value - current_position_value
            if abs(trade_value) > 0.01:
                costs = self.calculate_transaction_costs(abs(trade_value))
                self.trades.append(
                    {
                        "Date": date,
                        "Execution_Price": execution_price,
                        "Trade_Value": trade_value,
                        "Costs": costs,
                        "Regime": current_regime,
                    }
                )
            else:
                costs = 0.0
            if abs(target_position_value) > 0.01:
                self.positions.iloc[i] = target_position_value / execution_price
            else:
                self.positions.iloc[i] = 0.0
            if abs(trade_value) > 0.01:
                if self.df["Close"].iloc[i - 1] > 0:
                    asset_return = self.df["Open"].iloc[i] / self.df["Close"].iloc[i - 1] - 1
                else:
                    asset_return = 0.0
            elif self.df["Close"].iloc[i - 1] > 0:
                asset_return = self.df["Close"].iloc[i] / self.df["Close"].iloc[i - 1] - 1
            else:
                asset_return = 0.0
            if abs(current_position_value) > 0.01:
                position_pnl = current_position_value * asset_return
            else:
                position_pnl = 0.0
            if self.config.COMPOUNDING_MODE:
                self.portfolio_values.iloc[i] = (
                    self.portfolio_values.iloc[i - 1] + position_pnl - costs
                )
            else:
                self.cash_reserves.iloc[i] = self.cash_reserves.iloc[i - 1] + position_pnl - costs
                self.portfolio_values.iloc[i] = (
                    self.config.INITIAL_CAPITAL + self.cash_reserves.iloc[i]
                )
        results = pd.DataFrame(
            {
                "Date": self.df["Date"],
                "Adj_Close": self.df["Adj_Close"],
                "Position": self.positions,
                "Portfolio_Value": self.portfolio_values,
                "Cash_Reserve": self.cash_reserves,
                "Regime": self.regime_history,
            }
        )
        results["Strategy_Return"] = results["Portfolio_Value"].pct_change()
        results["Asset_Return"] = results["Adj_Close"].pct_change()
        print(f"✓ Backtest complete: {len(self.trades)} trades executed")
        return results


class RobustnessMetrics:
    """
    Comprehensive statistical validation metrics for strategy robustness.

    Implements advanced metrics to detect overfitting and validate
    statistical significance of backtest results:

    - Deflated Sharpe Ratio (DSR): Adjusts for multiple testing bias
    - Probabilistic Sharpe Ratio (PSR): Tests significance vs benchmark
    - Walk-Forward Efficiency (WFE): Compares IS vs OOS performance
    - Hurst Exponent: Detects mean-reversion vs trending vs random
    - CDaR: Conditional Drawdown at Risk (tail risk measure)
    - Parameter Stability Index: Tests sensitivity to hyperparameters
    - Global Robustness Index: Combined normalized score

    Reference: Bailey et al. (2014-2017), Multiple papers on backtest overfitting
    """

    @staticmethod
    def calculate_sharpe_ratio(returns: pd.Series, periods_per_year: int = 252) -> float:
        """Calculate annualized Sharpe ratio."""
        if len(returns) < 2:
            return 0.0
        mean_return = returns.mean() * periods_per_year
        std_return = returns.std() * np.sqrt(periods_per_year)
        if std_return == 0:
            return 0.0
        return mean_return / std_return

    @staticmethod
    def probabilistic_sharpe_ratio(
        returns: pd.Series, sharpe_target: float = 1.0, periods_per_year: int = 252
    ) -> float:
        """
        Probabilistic Sharpe Ratio (PSR) - Bailey & López de Prado (2012)

        Tests the probability that observed Sharpe ratio exceeds a target.
        Accounts for return skewness and kurtosis.

        Args:
            returns: Daily returns
            sharpe_target: Target Sharpe ratio to test against (typically 1.0)
            periods_per_year: Annualization factor

        Returns:
            Probability that true Sharpe > target (0 to 1)
        """
        n = len(returns)
        if n < 2:
            return 0.0
        sharpe_obs = RobustnessMetrics.calculate_sharpe_ratio(returns, periods_per_year)
        skew = returns.skew()
        kurt = returns.kurtosis()
        sharpe_std = np.sqrt(
            (1 + sharpe_obs**2 / 2 - skew * sharpe_obs + (kurt - 3) / 4 * sharpe_obs**2) / (n - 1)
        )
        z_score = (sharpe_obs - sharpe_target) / sharpe_std
        psr = stats.norm.cdf(z_score)
        return psr

    @staticmethod
    def deflated_sharpe_ratio(
        returns: pd.Series, n_trials: int = 100, periods_per_year: int = 252
    ) -> float:
        """
        Deflated Sharpe Ratio (DSR) - Bailey & López de Prado (2014)

        Adjusts Sharpe ratio for multiple testing bias (trying many strategies).
        Uses PSR framework with adjustment for number of trials.

        Args:
            returns: Daily returns
            n_trials: Number of strategies tested (typically 100+)
            periods_per_year: Annualization factor

        Returns:
            Deflated Sharpe ratio (accounts for multiple testing)
        """
        n = len(returns)
        if n < 2:
            return 0.0
        sharpe_obs = RobustnessMetrics.calculate_sharpe_ratio(returns, periods_per_year)
        gamma = 0.5772156649
        expected_max_sharpe = (1 - gamma) * stats.norm.ppf(
            1 - 1 / n_trials
        ) + gamma * stats.norm.ppf(1 - 1 / (n_trials * np.e))
        skew = returns.skew()
        kurt = returns.kurtosis()
        sharpe_std = np.sqrt(
            (1 + sharpe_obs**2 / 2 - skew * sharpe_obs + (kurt - 3) / 4 * sharpe_obs**2) / (n - 1)
        )
        z_score = (sharpe_obs - expected_max_sharpe) / sharpe_std
        dsr = stats.norm.cdf(z_score)
        return dsr

    @staticmethod
    def walk_forward_efficiency(sharpe_is: float, sharpe_oos: float) -> float:
        """
        Walk-Forward Efficiency (WFE)

        Compares in-sample vs out-of-sample Sharpe ratio.
        WFE < 0.5 suggests overfitting.

        Args:
            sharpe_is: In-sample Sharpe ratio
            sharpe_oos: Out-of-sample Sharpe ratio

        Returns:
            Efficiency ratio (0 to 1+)
        """
        if abs(sharpe_is) < 0.01:
            return 0.0
        wfe = sharpe_oos / sharpe_is
        return max(0.0, wfe)

    @staticmethod
    def hurst_exponent(returns: pd.Series, lag: int = 20) -> float:
        """
        Hurst Exponent - Tests for mean-reversion, random walk, or trending.

        H < 0.5: Mean-reverting (anti-persistent)
        H = 0.5: Random walk (geometric Brownian motion)
        H > 0.5: Trending (persistent)

        Args:
            returns: Return series
            lag: Maximum lag for R/S analysis

        Returns:
            Hurst exponent (0 to 1)
        """
        if len(returns) < lag * 2:
            return 0.5
        returns_clean = returns.dropna()
        if len(returns_clean) < lag * 2:
            return 0.5
        lags = range(2, lag)
        tau = []
        rs_values = []
        for lag_val in lags:
            n_chunks = len(returns_clean) // lag_val
            if n_chunks == 0:
                continue
            rs_chunk = []
            for i in range(n_chunks):
                chunk = returns_clean.iloc[i * lag_val : (i + 1) * lag_val]
                mean_val = chunk.mean()
                cumsum = (chunk - mean_val).cumsum()
                R = cumsum.max() - cumsum.min()
                S = chunk.std()
                if S > 0:
                    rs_chunk.append(R / S)
            if len(rs_chunk) > 0:
                tau.append(lag_val)
                rs_values.append(np.mean(rs_chunk))
        if len(tau) < 2:
            return 0.5
        tau = np.array(tau)
        rs_values = np.array(rs_values)
        valid = (tau > 0) & (rs_values > 0)
        if valid.sum() < 2:
            return 0.5
        tau = tau[valid]
        rs_values = rs_values[valid]
        log_tau = np.log(tau)
        log_rs = np.log(rs_values)
        hurst = np.polyfit(log_tau, log_rs, 1)[0]
        hurst = np.clip(hurst, 0.0, 1.0)
        return hurst

    @staticmethod
    def conditional_drawdown_at_risk(returns: pd.Series, confidence: float = 0.95) -> float:
        """
        Conditional Drawdown at Risk (CDaR)

        Average of worst drawdowns beyond the confidence level.
        More comprehensive than maximum drawdown alone.

        Args:
            returns: Return series
            confidence: Confidence level (typically 0.95)

        Returns:
            CDaR as fraction of portfolio (negative value)
        """
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        sorted_dd = drawdown.sort_values()
        n_worst = max(1, int(len(sorted_dd) * (1 - confidence)))
        worst_dds = sorted_dd.iloc[:n_worst]
        cdar = worst_dds.mean()
        return cdar

    @staticmethod
    def global_robustness_index(metrics: Dict[str, float]) -> float:
        """
        Global Robustness Index (GRI)

        Combines multiple robustness metrics into single normalized score.
        GRI > 0.65 suggests strategy is statistically robust.

        Metrics included:
        - PSR (weight: 0.25)
        - DSR (weight: 0.25)
        - WFE (weight: 0.20)
        - Hurst deviation from 0.5 (weight: 0.15)
        - CDaR normalized (weight: 0.15)

        Args:
            metrics: Dictionary of robustness metrics

        Returns:
            Normalized GRI score (0 to 1)
        """
        psr = metrics.get("PSR", 0.0)
        dsr = metrics.get("DSR", 0.0)
        wfe = metrics.get("WFE", 0.0)
        hurst = metrics.get("Hurst", 0.5)
        cdar = metrics.get("CDaR", -1.0)
        psr_norm = np.clip(psr, 0, 1)
        dsr_norm = np.clip(dsr, 0, 1)
        wfe_norm = np.clip(wfe, 0, 1)
        hurst_score = 1 - abs(hurst - 0.55) * 2
        hurst_norm = np.clip(hurst_score, 0, 1)
        cdar_norm = np.clip(1 + cdar / 0.5, 0, 1)
        weights = {"PSR": 0.25, "DSR": 0.25, "WFE": 0.2, "Hurst": 0.15, "CDaR": 0.15}
        gri = (
            weights["PSR"] * psr_norm
            + weights["DSR"] * dsr_norm
            + weights["WFE"] * wfe_norm
            + weights["Hurst"] * hurst_norm
            + weights["CDaR"] * cdar_norm
        )
        return gri


class PerformanceAnalytics:
    """
    Comprehensive performance analysis and visualization.

    Calculates standard metrics (Sharpe, Sortino, Max DD) and
    advanced robustness metrics (PSR, DSR, WFE, Hurst, CDaR, GRI).
    """

    @staticmethod
    def calculate_metrics(results: pd.DataFrame, benchmark_df: pd.DataFrame = None) -> Dict:
        """
        Calculate comprehensive performance metrics.

        Args:
            results: Backtest results DataFrame
            benchmark_df: Benchmark price data (optional)

        Returns:
            Dictionary of performance metrics
        """
        returns = results["Strategy_Return"].dropna()
        total_return = results["Portfolio_Value"].iloc[-1] / config.INITIAL_CAPITAL - 1
        ann_return = (1 + total_return) ** (252 / len(returns)) - 1
        sharpe = RobustnessMetrics.calculate_sharpe_ratio(returns)
        downside_returns = returns[returns < 0]
        if len(downside_returns) > 0:
            downside_std = downside_returns.std() * np.sqrt(252)
            sortino = ann_return / downside_std if downside_std > 0 else 0.0
        else:
            sortino = 0.0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        calmar = abs(ann_return / max_dd) if max_dd != 0 else 0.0
        win_rate = (returns > 0).sum() / len(returns)
        psr = RobustnessMetrics.probabilistic_sharpe_ratio(returns, config.PSR_SHARPE_TARGET)
        dsr = RobustnessMetrics.deflated_sharpe_ratio(returns, config.DSR_TRIALS)
        hurst = RobustnessMetrics.hurst_exponent(returns, config.HURST_LAG)
        cdar = RobustnessMetrics.conditional_drawdown_at_risk(returns, config.CDAR_CONFIDENCE)
        robustness_metrics = {"PSR": psr, "DSR": dsr, "WFE": 0.0, "Hurst": hurst, "CDaR": cdar}
        gri = RobustnessMetrics.global_robustness_index(robustness_metrics)
        metrics = {
            "Total_Return": total_return,
            "Annual_Return": ann_return,
            "Sharpe_Ratio": sharpe,
            "Sortino_Ratio": sortino,
            "Calmar_Ratio": calmar,
            "Max_Drawdown": max_dd,
            "Win_Rate": win_rate,
            "PSR": psr,
            "DSR": dsr,
            "Hurst": hurst,
            "CDaR": cdar,
            "GRI": gri,
        }
        if benchmark_df is not None:
            benchmark_returns = benchmark_df.set_index("Date")["Adj_Close"].pct_change()
            aligned_benchmark = benchmark_returns.reindex(results["Date"]).fillna(0)
            bench_total = (aligned_benchmark + 1).prod() - 1
            bench_ann = (1 + bench_total) ** (252 / len(aligned_benchmark)) - 1
            bench_sharpe = RobustnessMetrics.calculate_sharpe_ratio(aligned_benchmark)
            excess_returns = returns - aligned_benchmark
            tracking_error = excess_returns.std() * np.sqrt(252)
            if tracking_error > 0:
                information_ratio = (ann_return - bench_ann) / tracking_error
            else:
                information_ratio = 0.0
            metrics["Benchmark_Return"] = bench_total
            metrics["Benchmark_Sharpe"] = bench_sharpe
            metrics["Information_Ratio"] = information_ratio
        return metrics

    @staticmethod
    def plot_performance(results: pd.DataFrame, metrics: Dict, benchmark_df: pd.DataFrame = None):
        """
        Create comprehensive performance visualization.

        Args:
            results: Backtest results
            metrics: Performance metrics dictionary
            benchmark_df: Benchmark data (optional)
        """
        plt.style.use(config.PLOT_STYLE)
        fig = plt.figure(figsize=config.PLOT_FIGSIZE, dpi=config.PLOT_DPI)
        gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
        ax1 = fig.add_subplot(gs[0, :])
        norm_portfolio = results["Portfolio_Value"] / config.INITIAL_CAPITAL
        ax1.plot(results["Date"], norm_portfolio, label="Strategy", linewidth=2, color="darkblue")
        if benchmark_df is not None:
            bench_aligned = benchmark_df.set_index("Date")["Adj_Close"].reindex(results["Date"])
            norm_bench = bench_aligned / bench_aligned.iloc[0]
            ax1.plot(
                results["Date"],
                norm_bench,
                label=f"Benchmark ({config.BENCHMARK})",
                linewidth=2,
                color="gray",
                alpha=0.7,
            )
        ax1.set_title(
            f"{config.TICKER} YZRV HAR-GARCH Regime Strategy", fontsize=14, fontweight="bold"
        )
        ax1.set_ylabel("Normalized Value", fontsize=11)
        ax1.legend(loc="upper left", fontsize=10)
        ax1.grid(True, alpha=0.3)
        ax2 = fig.add_subplot(gs[1, 0])
        returns = results["Strategy_Return"].dropna()
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        ax2.fill_between(
            results["Date"].iloc[1:], drawdown * 100, 0, color="red", alpha=0.3, label="Drawdown"
        )
        ax2.set_title("Strategy Drawdown", fontsize=12, fontweight="bold")
        ax2.set_ylabel("Drawdown (%)", fontsize=10)
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=9)
        ax3 = fig.add_subplot(gs[1, 1])
        regime_counts = results["Regime"].value_counts()
        colors = ["green", "red"]
        ax3.bar(
            ["LOW_VOL", "HIGH_VOL"],
            [regime_counts.get(0, 0), regime_counts.get(1, 0)],
            color=colors,
            alpha=0.7,
        )
        ax3.set_title("Regime Distribution", fontsize=12, fontweight="bold")
        ax3.set_ylabel("Days", fontsize=10)
        ax3.grid(True, alpha=0.3, axis="y")
        ax4 = fig.add_subplot(gs[2, 0])
        rolling_sharpe = returns.rolling(window=60).apply(
            lambda x: RobustnessMetrics.calculate_sharpe_ratio(x) if len(x) == 60 else np.nan
        )
        ax4.plot(results["Date"].iloc[1:], rolling_sharpe, linewidth=1.5, color="purple")
        ax4.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
        ax4.set_title("Rolling 60-Day Sharpe Ratio", fontsize=12, fontweight="bold")
        ax4.set_ylabel("Sharpe", fontsize=10)
        ax4.grid(True, alpha=0.3)
        ax5 = fig.add_subplot(gs[2, 1])
        ax5.axis("off")
        metrics_text = f"\nPERFORMANCE METRICS\n{'=' * 40}\nTotal Return:        {metrics['Total_Return'] * 100:>8.2f}%\nAnnual Return:       {metrics['Annual_Return'] * 100:>8.2f}%\nSharpe Ratio:        {metrics['Sharpe_Ratio']:>8.3f}\nSortino Ratio:       {metrics['Sortino_Ratio']:>8.3f}\nCalmar Ratio:        {metrics['Calmar_Ratio']:>8.3f}\nMax Drawdown:        {metrics['Max_Drawdown'] * 100:>8.2f}%\nWin Rate:            {metrics['Win_Rate'] * 100:>8.2f}%\n\nROBUSTNESS METRICS\n{'=' * 40}\nPSR (vs {config.PSR_SHARPE_TARGET:.1f}):      {metrics['PSR']:>8.3f}\nDSR ({config.DSR_TRIALS} trials):   {metrics['DSR']:>8.3f}\nHurst Exponent:      {metrics['Hurst']:>8.3f}\nCDaR (95%):          {metrics['CDaR'] * 100:>8.2f}%\nGlobal Robust. Idx:  {metrics['GRI']:>8.3f}\n        "
        if "Benchmark_Return" in metrics:
            metrics_text += f"\nBENCHMARK COMPARISON\n{'=' * 40}\nBenchmark Return:    {metrics['Benchmark_Return'] * 100:>8.2f}%\nInformation Ratio:   {metrics['Information_Ratio']:>8.3f}\n            "
        ax5.text(
            0.1,
            0.95,
            metrics_text,
            transform=ax5.transAxes,
            fontsize=9,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3),
        )
        plt.savefig(
            "results/multi-feature-volatility-regimes/spy_yzrv_har_garch_performance.png",
            dpi=config.PLOT_DPI,
            bbox_inches="tight",
        )
        print("✓ Performance chart saved")


class WalkForwardValidator:
    """
    Walk-forward expanding window validation.

    Simulates production deployment with periodic model refitting:
    - Initial training period: 1 year
    - Refit frequency: Quarterly (63 days)
    - Window type: Expanding (not rolling)

    This prevents look-ahead bias and tests model stability over time.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.fold_results = []

    def run_validation(self, df: pd.DataFrame) -> Tuple[pd.Series, pd.Series, Dict]:
        """
        Run walk-forward validation with expanding windows.

        Args:
            df: Full dataset with OHLC prices

        Returns:
            Tuple of (signals, regimes, wf_metrics)
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD EXPANDING WINDOW VALIDATION")
        print("=" * 80)
        signals = pd.Series(0.0, index=df.index)
        regimes = pd.Series(0, index=df.index)
        yzrv_calculator = YangZhangVolatility()
        yzrv = yzrv_calculator.calculate_yzrv(
            df, window=config.YZRV_WINDOW, annualization=config.YZRV_ANNUALIZATION
        )
        har_model = HARGARCHModel(p=config.GARCH_P, q=config.GARCH_Q)
        har_features = har_model.create_har_features(
            yzrv,
            daily_window=config.HAR_DAILY_WINDOW,
            weekly_window=config.HAR_WEEKLY_WINDOW,
            monthly_window=config.HAR_MONTHLY_WINDOW,
        )
        df["Returns"] = df["Adj_Close"].pct_change()
        regime_detector = VolatilityRegimeDetector(vol_threshold=config.REGIME_VOL_THRESHOLD)
        train_start_idx = config.FEATURE_WARMUP_DAYS
        current_idx = train_start_idx + config.TRAIN_PERIOD_DAYS
        fold_num = 0
        while current_idx < len(df):
            fold_num += 1
            train_returns = df["Returns"].iloc[train_start_idx:current_idx]
            train_har = har_features.iloc[train_start_idx:current_idx]
            val_end_idx = min(current_idx + config.REFIT_FREQUENCY_DAYS, len(df))
            print(f"\nFold {fold_num}:")
            print(
                f"  Training:   {df['Date'].iloc[train_start_idx]} to {df['Date'].iloc[current_idx - 1]} ({current_idx - train_start_idx} days)"
            )
            print(
                f"  Validation: {df['Date'].iloc[current_idx]} to {df['Date'].iloc[val_end_idx - 1]} ({val_end_idx - current_idx} days)"
            )
            try:
                har_garch = HARGARCHModel(p=config.GARCH_P, q=config.GARCH_Q)
                har_garch.fit(train_returns, train_har)
                for val_idx in range(current_idx, val_end_idx):
                    current_yzrv = yzrv.iloc[val_idx - 1]
                    if pd.notna(current_yzrv):
                        regime = regime_detector.detect_regime(current_yzrv)
                        regimes.iloc[val_idx] = regime
                        allocation = regime_detector.get_regime_allocation(regime)
                        signals.iloc[val_idx] = allocation
                is_returns = train_returns.dropna()
                is_sharpe = RobustnessMetrics.calculate_sharpe_ratio(is_returns)
                oos_signals = signals.iloc[current_idx:val_end_idx]
                oos_returns = df["Returns"].iloc[current_idx:val_end_idx]
                oos_strategy_returns = oos_returns * oos_signals.shift(1)
                oos_sharpe = RobustnessMetrics.calculate_sharpe_ratio(oos_strategy_returns.dropna())
                wfe = RobustnessMetrics.walk_forward_efficiency(is_sharpe, oos_sharpe)
                self.fold_results.append(
                    {
                        "Fold": fold_num,
                        "Train_Start": df["Date"].iloc[train_start_idx],
                        "Train_End": df["Date"].iloc[current_idx - 1],
                        "Val_Start": df["Date"].iloc[current_idx],
                        "Val_End": df["Date"].iloc[val_end_idx - 1],
                        "IS_Sharpe": is_sharpe,
                        "OOS_Sharpe": oos_sharpe,
                        "WFE": wfe,
                    }
                )
                print(f"  IS Sharpe:  {is_sharpe:.3f}")
                print(f"  OOS Sharpe: {oos_sharpe:.3f}")
                print(f"  WFE:        {wfe:.3f}")
            except Exception as e:
                print(f"  ⚠ Fold {fold_num} failed: {str(e)}")
            current_idx = val_end_idx
        if len(self.fold_results) > 0:
            fold_df = pd.DataFrame(self.fold_results)
            avg_is_sharpe = fold_df["IS_Sharpe"].mean()
            avg_oos_sharpe = fold_df["OOS_Sharpe"].mean()
            avg_wfe = fold_df["WFE"].mean()
            wf_metrics = {
                "Avg_IS_Sharpe": avg_is_sharpe,
                "Avg_OOS_Sharpe": avg_oos_sharpe,
                "Avg_WFE": avg_wfe,
                "N_Folds": len(self.fold_results),
            }
            print("\n" + "=" * 80)
            print(f"WALK-FORWARD SUMMARY ({len(self.fold_results)} folds)")
            print("=" * 80)
            print(f"Average IS Sharpe:  {avg_is_sharpe:.3f}")
            print(f"Average OOS Sharpe: {avg_oos_sharpe:.3f}")
            print(f"Average WFE:        {avg_wfe:.3f}")
            print("=" * 80)
        else:
            wf_metrics = {}
        return (signals, regimes, wf_metrics)


def main():
    Path("results/multi-feature-volatility-regimes").mkdir(parents=True, exist_ok=True)
    """
    Main execution function for SPY YZRV HAR-GARCH regime strategy.
    """
    print("=" * 80)
    print("SPY VOLATILITY REGIME STRATEGY - YZRV + HAR-GARCH-X")
    print("=" * 80)
    print(f"\nConfiguration:")
    print(f"  Ticker:              {config.TICKER}")
    print(f"  Benchmark:           {config.BENCHMARK}")
    print(f"  Period:              {config.START_DATE} to {config.END_DATE}")
    print(f"  Initial Capital:     ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Compounding Mode:    {config.COMPOUNDING_MODE}")
    print(f"  Leverage:            {config.LEVERAGE}x")
    print(f"  Commission:          {config.COMMISSION_BPS} bps")
    print(f"  YZRV Window:         {config.YZRV_WINDOW} days")
    print(f"  Regime Threshold:    {config.REGIME_VOL_THRESHOLD * 100:.0f}% annualized vol")
    print(f"  Feature Warm-up:     {config.FEATURE_WARMUP_DAYS} days")
    print("=" * 80)
    fetcher = YahooDataFetcher()
    try:
        spy_data = fetcher.fetch_historical_data(config.TICKER, config.START_DATE, config.END_DATE)
        benchmark_data = fetcher.fetch_historical_data(
            config.BENCHMARK, config.START_DATE, config.END_DATE
        )
    except Exception as e:
        print(f"✗ Data fetch failed: {str(e)}")
        return
    wf_validator = WalkForwardValidator(config)
    signals, regimes, wf_metrics = wf_validator.run_validation(spy_data)
    backtester = ProductionBacktester(spy_data, config)
    results = backtester.run_backtest(signals, regimes)
    analytics = PerformanceAnalytics()
    metrics = analytics.calculate_metrics(results, benchmark_data)
    if wf_metrics:
        metrics["WFE"] = wf_metrics["Avg_WFE"]
        robustness_metrics = {
            "PSR": metrics["PSR"],
            "DSR": metrics["DSR"],
            "WFE": metrics["WFE"],
            "Hurst": metrics["Hurst"],
            "CDaR": metrics["CDaR"],
        }
        metrics["GRI"] = RobustnessMetrics.global_robustness_index(robustness_metrics)
    print("\n" + "=" * 80)
    print("FINAL PERFORMANCE RESULTS")
    print("=" * 80)
    print(f"Total Return:          {metrics['Total_Return'] * 100:>7.2f}%")
    print(f"Annualized Return:     {metrics['Annual_Return'] * 100:>7.2f}%")
    print(f"Sharpe Ratio:          {metrics['Sharpe_Ratio']:>7.3f}")
    print(f"Sortino Ratio:         {metrics['Sortino_Ratio']:>7.3f}")
    print(f"Max Drawdown:          {metrics['Max_Drawdown'] * 100:>7.2f}%")
    print(f"Win Rate:              {metrics['Win_Rate'] * 100:>7.2f}%")
    print(f"\nROBUSTNESS METRICS:")
    print(f"PSR:                   {metrics['PSR']:>7.3f}")
    print(f"DSR:                   {metrics['DSR']:>7.3f}")
    if "WFE" in metrics:
        print(f"WFE:                   {metrics['WFE']:>7.3f}")
    print(f"Hurst:                 {metrics['Hurst']:>7.3f}")
    print(f"CDaR:                  {metrics['CDaR'] * 100:>7.2f}%")
    print(f"GRI:                   {metrics['GRI']:>7.3f}")
    if "Benchmark_Return" in metrics:
        print(f"\nBENCHMARK COMPARISON:")
        print(f"Benchmark Return:      {metrics['Benchmark_Return'] * 100:>7.2f}%")
        print(f"Information Ratio:     {metrics['Information_Ratio']:>7.3f}")
    print("=" * 80)
    print("\nINTERPRETATION:")
    if metrics["GRI"] > 0.65:
        print("✓ Strategy shows STRONG statistical robustness (GRI > 0.65)")
    elif metrics["GRI"] > 0.5:
        print("~ Strategy shows MODERATE robustness (GRI 0.50-0.65)")
    else:
        print("✗ Strategy shows WEAK robustness (GRI < 0.50) - requires improvement")
    if "WFE" in metrics:
        if metrics["WFE"] > 0.5:
            print("✓ Walk-forward efficiency > 0.5: Low overfitting risk")
        else:
            print("✗ Walk-forward efficiency < 0.5: Potential overfitting detected")
    if 0.45 < metrics["Hurst"] < 0.6:
        print("✓ Hurst exponent near 0.5: Returns close to random walk (good)")
    elif metrics["Hurst"] > 0.6:
        print("~ Hurst > 0.6: Strategy captures trending behavior")
    else:
        print("~ Hurst < 0.45: Strategy captures mean-reversion")
    analytics.plot_performance(results, metrics, benchmark_data)
    results.to_csv(
        "results/multi-feature-volatility-regimes/spy_yzrv_har_garch_results.csv", index=False
    )
    print("\n✓ Results exported to CSV")
    if wf_validator.fold_results:
        fold_df = pd.DataFrame(wf_validator.fold_results)
        fold_df.to_csv(
            "results/multi-feature-volatility-regimes/spy_yzrv_har_garch_walkforward.csv",
            index=False,
        )
        print("✓ Walk-forward details exported to CSV")
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    main()
